# Итоговый проект. Отток клиентов телеком-оператора — ОТЧЁТ

**Студент:** (ФИО, группа) · **Номер по журналу:** впишите ниже

Этот ноутбук — ваш отчёт. Выполните все ячейки сверху вниз (**Run All**) перед сдачей:
ноутбук сдаётся **вместе с результатами выполнения**. Задание и критерии — в `README_project.md`.

Разделы:
0. Паспорт (обязателен, выполняется первым)
1. Стек
2. Данные → S3
3. PostgreSQL
4. EDA и визуализация
5. Модели в MLflow
6. Конвейер в Airflow
7. Выводы и личные вопросы
8. Самопроверка (должна быть зелёной)


## 0. Паспорт

Номер `N` = ваш номер по журналу. Паспорт — хэш вашего личного датасета:
преподаватель перегенерирует данные по `N` и сверит хэш. Это защита от копирования.


In [ ]:
import hashlib
from pathlib import Path

STUDENT = "user1"          # логин в стеке (не менять)
N = 7                      # <-- ВАШ НОМЕР ПО ЖУРНАЛУ

# параметры варианта (формулы — в README_project.md)
TEST_SIZE = {0: 0.20, 1: 0.25, 2: 0.30}[N % 3]
EXTRA_MODEL = ["HistGradientBoostingClassifier", "SVC(probability=True)",
               "GaussianNB", "ExtraTreesClassifier"][N % 4]
PRIMARY_METRIC = "roc_auc" if N % 2 == 0 else "average_precision"
P_FLAKY = round(0.25 + 0.10 * (N % 3), 2)
GATE_MARGIN = 0.01

EXPERIMENT = f"user1_p{N}_churn"
MODEL_NAME = f"user1_p{N}_model"
DAG_ID = f"user1_p{N}_churn"

print(f"N={N} | тест={TEST_SIZE} | доп.модель={EXTRA_MODEL} | метрика={PRIMARY_METRIC}")
print(f"вероятность сбоя DAG p={P_FLAKY} | порог ворот=чемпион+{GATE_MARGIN}")

raw_path = Path(f"customers_{N}.csv")
assert raw_path.exists(), f"сначала сгенерируйте датасет: python make_dataset.py --student {N}"
PASSPORT = hashlib.sha256(f"rudn2026:{N}:".encode() + raw_path.read_bytes()).hexdigest()[:16]
print("PASSPORT:", PASSPORT)


> **PASSPORT** запомните — он печатается здесь и проверяется в конце (раздел 8).


## 1. Стек (1 балл)

Выполняется в Jupyter стека (localhost:8888). Проверяем, что все сервисы живые.


In [ ]:
import os
import boto3
import mlflow
import psycopg2
import requests

def check(name, fn):
    try:
        fn(); print(f"  [OK] {name}")
    except Exception as e:
        print(f"  [FAIL] {name}: {type(e).__name__} {e}")

check("S3", lambda: boto3.client("s3", endpoint_url=os.environ["S3_ENDPOINT_URL"],
        aws_access_key_id=STUDENT, aws_secret_access_key=os.environ["S3_PASSWORD"]).head_bucket(Bucket=STUDENT))
check("MLflow", lambda: mlflow.search_experiments(max_results=1))
check("PostgreSQL", lambda: psycopg2.connect(host="postgres", dbname="postgres",
        user="airflow", password="airflow", connect_timeout=5).close())
check("Airflow API", lambda: requests.get("http://airflow-webserver:8080/api/v1/health",
        auth=("student", "student"), timeout=5).raise_for_status())


## 2. Персональные данные → S3 (2 балла)

1. Сгенерируйте датасет (в ячейке ниже) и посмотрите на него.
2. Загрузите сырой CSV в S3: `project/{N}/raw/customers_{N}.csv`.
3. Опишите в маркдауне, какие **3 проблемы** вы нашли в данных (дубликаты? пропуски? выбросы?)
   и какая колонка — **утечка** (почему её нельзя использовать при обучении).


In [ ]:
!python make_dataset.py --student {N}


In [ ]:
import pandas as pd

df = pd.read_csv(f"customers_{N}.csv")
print(df.shape)
df.head()


In [ ]:
# TODO: загрузить сырой CSV в S3 по ключу project/{N}/raw/customers_{N}.csv
# (boto3, бакет STUDENT — как в практикуме 1) и показать ls("project/{N}/")
...


**Найденные проблемы и колонка-утечка:** (ваш текст)


## 3. Данные в PostgreSQL (2 балла)

Создайте схему `user1_project`, таблицу `customers` (очищенные данные, без дубликатов),
загрузите строки и ответьте на 3 SQL-вопроса из задания: запрос + результат + вывод одной строкой.


In [ ]:
# TODO: подключение (host="postgres", dbname="postgres", user="airflow", password="airflow"),
# CREATE SCHEMA user1_project, CREATE TABLE ... customers, загрузка данных
...


In [ ]:
# TODO: SQL-вопрос 1 — доля оттока по типам контракта
...


In [ ]:
# TODO: SQL-вопрос 2 — топ-5 сегментов (интернет × способ оплаты) по доле оттока
...


In [ ]:
# TODO: SQL-вопрос 3 — среднее число звонков в поддержку: ушедшие vs оставшиеся
...


## 4. EDA и визуализация (3 балла)

Минимум **8 графиков**, у каждого: заголовок и **вывод** (1–2 предложения вашим текстом
под ячейкой). Каждый PNG сохранить в S3: `project/{N}/eda/<имя>.png`
(функция-помощник ниже уже умеет и показывать, и сохранять).


In [ ]:
import io
import matplotlib.pyplot as plt

def show_and_save(fig, name):
    '''Показывает график в ноутбуке и кладёт PNG в S3: project/{N}/eda/<name>.png'''
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    s3c = boto3.client("s3", endpoint_url=os.environ["S3_ENDPOINT_URL"],
                       aws_access_key_id=STUDENT, aws_secret_access_key=os.environ["S3_PASSWORD"])
    s3c.put_object(Bucket=STUDENT, Key=f"project/{N}/eda/{name}.png",
                   Body=buf.getvalue(), ContentType="image/png")
    return fig


In [ ]:
# TODO: график 1 — баланс классов (churn)
...


**Вывод по графику 1:** ...


In [ ]:
# TODO: графики 2–8: распределения по классам, корреляции, отток по категориям,
# отток от стажа, влияние support_calls, возраст — что считаете нужным.
# Каждый: fig = ...; show_and_save(fig, "имя"); под ячейкой — вывод.
...


## 5. Модели и реестр в MLflow (4 балла)

Эксперимент `user1_p{N}_churn`. Требования: ≥ 15 запусков (4 обязательные модели
+ модель по варианту, у каждой ≥ 3 значения гиперпараметра), в каждом — параметры,
обе метрики (`roc_auc` и `average_precision`), модель. Для лучшей — матрица ошибок,
ROC, PR, важность признаков как артефакты. Затем реестр + ворота качества.


In [ ]:
# TODO: подготовка данных (чистка как в части 2, разбиение с TEST_SIZE из паспорта,
# ColumnTransformer: OneHotEncoder + StandardScaler)
...


In [ ]:
# TODO: цикл обучения всех моделей с перебором параметров; каждый запуск:
# mlflow.start_run(run_name=...) -> log_params, log_metrics(roc_auc И average_precision),
# mlflow.sklearn.log_model (skops_trusted_types — см. практикум 2)
...


In [ ]:
# TODO: таблица сравнения (search_runs, сортировка по PRIMARY_METRIC), лучшая модель;
# для лучшей: матрица ошибок, ROC-кривая, PR-кривая, важность признаков -> log_figure
...


In [ ]:
# TODO: реестр: register_model + метка @champion;
# ворота качества: претендент (изменённый гиперпараметр) vs чемпион + GATE_MARGIN;
# показать оба числа и исход (переезд метки или отказ) — в лог и в отчёт
...


## 6. Конвейер в Airflow (5 баллов)

Ваш файл DAG: `user1_p{N}_churn.py` в папке `dags` (скопируйте `dag_template.py`
и заполните TODO). Требования из задания: ненадёжная `load_raw` с повторами,
параллельные `train`, `select_best`, ворота качества, регистрация чемпиона,
`top_risk` в S3 + PostgreSQL. Запустите DAG дважды.

Ниже — проверки, которые сами достанут факт повторов и статусы из Airflow.


In [ ]:
# последний запуск вашего DAG: статусы задач и число попыток
import requests

def airflow(path):
    r = requests.get(f"http://airflow-webserver:8080/api/v1{path}",
                     auth=("student", "student"), timeout=10)
    r.raise_for_status()
    return r.json()

runs = airflow(f"/dags/{DAG_ID}/dagRuns?order_by=-execution_date&limit=5")["dag_runs"]
assert runs, "запусков нет — сначала запустите DAG в Airflow"
print("запусков найдено:", len(runs), "| последний:", runs[0]["state"], runs[0]["execution_date"])

ti = airflow(f"/dags/{DAG_ID}/dagRuns/{runs[0]['dag_run_id']}/taskInstances")["task_instances"]
for t in sorted(ti, key=lambda x: x["task_id"]):
    print(f'  {t["task_id"]:<20} {t["state"]:<10} попыток: {t["try_number"]}')


**Что видно в Grid:** (ваш текст: где была оранжевая попытка, сколько попыток
понадобилось load_raw, чем закончился запуск; приложите скриншот Grid к сдаче)

**Второй запуск и ворота качества:** (ваш текст: чемпион удержал метку или нет и почему это правильно)


## 7. Выводы и личные вопросы (3 балла)

Ответьте своими словами — они проверяются индивидуально, у каждого данные разные.


1. **Какая модель победила на ваших данных и почему?** (сравните с соседом: у него может быть другая)
2. **Колонка-утечка:** какая и как вы её нашли?
3. **Повторы в Airflow:** сколько попыток понадобилось load_raw? Что было бы с cron-скриптом?
4. **Ворота качества:** ваш чемпион удержал метку? Почему отказ — это нормальный исход?
5. **Бизнес-вывод:** кому из клиентов (какому сегменту) компания должна позвонить первой и почему?


## 8. Самопроверка (автоматическая)

Ячейка сама проверяет ключевые требования. **Сдавать работу можно только с зелёной
самопроверкой** — если что-то красное, вернитесь к соответствующей части.


In [ ]:
import hashlib
from pathlib import Path

results = []
def ok(name, cond, detail=""):
    results.append((name, bool(cond), detail))

# 1) паспорт совпадает с файлом датасета
raw = Path(f"customers_{N}.csv")
if raw.exists():
    h = hashlib.sha256(f"rudn2026:{N}:".encode() + raw.read_bytes()).hexdigest()[:16]
    ok("Паспорт датасета", h == PASSPORT, h)
else:
    ok("Паспорт датасета", False, "файл датасета не найден")

# 2) сырые данные и графики EDA в S3
s3c = boto3.client("s3", endpoint_url=os.environ["S3_ENDPOINT_URL"],
                   aws_access_key_id=STUDENT, aws_secret_access_key=os.environ["S3_PASSWORD"])
def keys(prefix):
    return [o["Key"] for o in s3c.list_objects_v2(Bucket=STUDENT, Prefix=prefix).get("Contents", [])]
ok("Сырой CSV в S3", any(k.endswith(f"customers_{N}.csv") for k in keys(f"project/{N}/raw/")))
ok("≥8 графиков EDA в S3", len(keys(f"project/{N}/eda/")) >= 8, f'{len(keys(f"project/{N}/eda/"))} шт')

# 3) MLflow: запуски и чемпион
runs_df = mlflow.search_runs(experiment_names=[EXPERIMENT])
ok("≥15 запусков в эксперименте", len(runs_df) >= 15, f"{len(runs_df)} запусков")
ok("≥5 разных моделей", runs_df["tags.mlflow.runName"].str.split("_").str[0].nunique() >= 5)
try:
    mv = mlflow.MlflowClient().get_model_version_by_alias(MODEL_NAME, "champion")
    ok("Чемпион в реестре", True, f"версия {mv.version}")
except Exception:
    ok("Чемпион в реестре", False, "метка @champion не найдена")

# 4) Airflow: последний запуск зелёный и были повторы
try:
    last = airflow(f"/dags/{DAG_ID}/dagRuns?order_by=-execution_date&limit=1")["dag_runs"][0]
    ti = airflow(f"/dags/{DAG_ID}/dagRuns/{last['dag_run_id']}/taskInstances")["task_instances"]
    ok("Последний запуск DAG успешен", last["state"] == "success", last["execution_date"])
    retries = max((t["try_number"] for t in ti), default=0)
    ok("Показан повтор (try_number ≥ 2)", retries >= 2, f"макс. попыток: {retries}")
    ok("top_risk сохранил CSV", any(k.endswith("top_risk.csv") for k in keys(f"project/{N}/")))
except Exception as e:
    ok("Airflow", False, str(e)[:80])

# 5) PostgreSQL: таблица top_risk не пустая
try:
    conn = psycopg2.connect(host="postgres", dbname="postgres", user="airflow", password="airflow")
    cnt = pd.read_sql(f'SELECT count(*) FROM user1_project.top_risk', conn).iloc[0, 0]
    conn.close()
    ok("top_risk в PostgreSQL", cnt >= 50, f"{cnt} строк")
except Exception as e:
    ok("top_risk в PostgreSQL", False, str(e)[:80])

print()
print("=" * 62)
n_ok = sum(1 for _, c, _ in results if c)
for name, cond, detail in results:
    print(f'  [{"OK" if cond else "FAIL"}] {name}' + (f' — {detail}' if detail else ""))
print("=" * 62)
print(f"ИТОГ: {n_ok}/{len(results)} проверок пройдено")
assert n_ok == len(results), "есть красные проверки — доработайте части проекта"


---

**Сдача:** этот ноутбук (.ipynb с выполненными ячейками) + `user1_p{N}_churn.py` +
скриншот Grid Airflow с повтором задачи. Отправьте преподавателю файлы одним архивом.
